In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GridSearchCV, train_test_split

In [2]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

e:\Anaconda\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [3]:
img_width = (224, 224)
batch_size = 32

datagen = ImageDataGenerator(rescale=1./255)

train_data = datagen.flow_from_directory(
    'dataset/train/train',
    target_size = img_width,
    batch_size = batch_size,
    class_mode = "categorical"
)

val_data = datagen.flow_from_directory(
    'dataset/val',
    target_size = img_width,
    batch_size = batch_size,
    class_mode = "categorical"
)

Found 16854 images belonging to 33 classes.
Found 16120 images belonging to 33 classes.


In [4]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import models, layers

In [8]:
base_model = MobileNetV2(
    input_shape = (224, 224, 3),
    include_top = False,
    weights = "imagenet"
)

base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation="relu"),
    layers.Dense(train_data.num_classes, activation="softmax")
])

model.compile(
    optimizer = "adam",
    loss = "categorical_crossentropy",
    metrics = ["accuracy"]
)

model.fit(train_data, epochs=2, validation_data = val_data)

Epoch 1/2
 22/527 ━━━━━━━━━━━━━━━━━━━━ 1:59 237ms/step - accuracy: 0.2886 - loss: 2.9230

KeyboardInterrupt: 

In [7]:
from tensorflow.keras.preprocessing import image

img = image.load_img("download-1-600x600.jpg", target_size=(224,224))
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

prediction = model.predict(img_array)
print(prediction)

pred_class = np.argmax(prediction, axis=1)
print("Predicted class index:", pred_class[0])

class_names = list(train_data.class_indices.keys())
print("Predicted class:", class_names[pred_class[0]])

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 852ms/step
[[1.5177238e-01 3.3610128e-04 5.0136563e-03 1.7723162e-04 8.7043904e-03
  3.8867159e-04 3.4360652e-04 7.6971189e-03 1.0356415e-03 1.5824403e-01
  1.6380752e-03 1.1287843e-02 7.6892749e-05 2.1625987e-04 3.0551222e-05
  9.9747558e-05 5.1374249e-05 8.7060110e-04 1.6098345e-02 5.5613433e-05
  1.7388983e-02 6.9759614e-03 1.1558379e-01 1.8239748e-02 9.7273253e-02
  9.7480707e-04 1.5296119e-02 1.7112239e-01 5.5057099e-03 1.0594364e-01
  5.2436903e-02 2.2393374e-02 6.7271283e-03]]
Predicted class index: 27
Predicted class: Pomegranate
